# Распознавание и синтез речи (Crash Course)

Основной акцент сделаем на извлечение признаков из аудиоданных и возможности применения известных вам алгоритмов.

Работа с аудиоданными с одной стороны сильно напоминает использование методов NLP, с другой - явно имеет много общего с компьютерным зрением.

**Замечание:** я отдаю себе отчет в том, что наибольший профит и понимание получат студенты, которые знакомы со следующими темами:
- сверточные нейронные сети;
- обработка последовательностей;
- рекуррентные нейронные сети;
- трансформеры.

Постараюсь без духоты, коллеги :)

## Знакомство с аудиоданными

**Звук - это волна**, которая характеризует периоды напряженности и разряженности среды (например воздуха).

![](https://i.imgur.com/djrpF8z.png)

Звуковая волна характеризуется:
- частотой ($f=\frac{1}{T}$, $T$ - период);
- амплитудой ($A$);
- фазой ($\phi$, актуально когда звуковые волны накладываются друг на друга).

![](https://i.imgur.com/AIn5vVS.png)

Звук сам по себе имеет непрерывную природу, однако рассматривая аудиоданные мы автоматически переходим в дискретное пространство, которое определяется **частотой дискретизации** (sample rate).

![](https://i.imgur.com/dozTHRE.png)

На сегодняшний день при хранении данных в сжатом формате MPEG-3 (mp3) мы часто сталкиваемся с частотой дискретизации 44100 Гц = 44100 (это означает, что одной секунде у нас соответствует 44100 точек на звуковой волне).

Таким образом аудиотрек **может интерпретироваться как временной ряд**. А временные ряды - это последовательности. Значит, мы будем иметь дело с обработкой последовательностей. Зафиксируем это.

# Какие признаки мы можем извлечь из аудиоданных (что чаще всего извлекается)

При извлечении признаков из аудиоданных мы рассматриваем не отдельные значения дискретизации, а **отдельные пересекающиеся фреймы** звукового сигнала:

![](https://i.imgur.com/diBBC6c.png)

Таким образом, появляется два параметра - размер фрейма (или размер окна) и длина прыжка (расстояние от начала первого окна до начала следующего). В результате извлечения признаков мы получаем последовательность значений этих самых признаков. Количество элементов последовательности, как правило, определяется количеством фреймов.

**Замечание:** предполагается, что все фреймы содержат одинаковое количество точек дискретизации.Если последний фрейм получается короче остальных - применяется **паддинг** (как правило, происходит заполнение нулями).

### Разреженное быстрое преобразование Фурье (SFFT)

Преобразование позволяет для каждого фрейма получить распределение интенсивности частотных полос.

![](https://i.imgur.com/Ya8jOBi.png)

При заданных параметрах результатом преобразования является матрица размерности (frequency bins, frames). В ячейках матрицы - интенсивность данной полосы в данном фрейме.

![](https://i.imgur.com/YSG7s99.png)

Более подробно про алгоритм можно узнать [здесь](https://habr.com/ru/companies/otus/articles/449996/).

### MEL-спектрограммы

Доказано, что человек воспринимает повышение частоты логарифмически. Поэтому очень часто значения частот преобразуют в MEL-шкалу (MEL от слова MELODY).

![](https://i.imgur.com/udlCOc9.png)

MEL-спектрограмма - это также матрица. Размерность (bands, frames). Заранее определяется количество частотных полос. Сначала вычисляется преобразование Фурье для указанного числа полос, после этого значения полос переводятся в MEL-шкалу по формуле выше.

**Замечание:** за счет того, что всегда можно выполнить обратное преобразование из MEL-шкалы, а также за счет того, что существует обратное преобразование Фурье, мы можем построить взаимно однозначное соответствие между множествами всех звуковых треков и всех MEL-спектрограмм.

Учитывая, что результаты быстрого преобразования Фурье и MEL-спектрограммы можно визуализировать (в виде heatmap например) **задача синтеза аудиофайлов часто сводится к задаче генерации изображений!** Это позволяет применять генеративно-состязательные нейронные сети.

### Мелкепстральные коэффициенты (я заставлял третьекурсников выговаривать это - одно из заданий ЛР)

Еще один способ извлечения признаков - получение значений мелкепстральных коэффициентов для каждого фрейма. И снова результатом является матрица. Размерность (n_coef, frames).

Мелкепстральные ккоэффициенты на практике извлекаются быстрее, чем выполняется дискретное преобразование Фурье. Алгоритм подробно описан [здесь](https://habr.com/ru/articles/140828/).

## Как использовать извлеченные признаки?

Мы видим, что признаки аудиоданных (акустические признаки) представляют собой **последовательности векторов**. Это значит, что для решения задач распознавания речи (например, классификации голосов) мы можем использовать **рекуррентные нейронные сети**.

Но берите выше - каждый фрейм связан с другими фреймами, он оказывает на них влияние. А это значит, что при работе с каждым фреймом мы должны **обращать внимание** на другие. В настоящее время для решения задач распознавания речи часто используют **трансформеры**.

![](https://media.springernature.com/lw685/springer-static/image/art%3A10.1038%2Fs41598-022-12260-y/MediaObjects/41598_2022_12260_Fig2_HTML.png)

Подробно об архитектуре Трансформер в приложении к NLP рассказано в [соответствующей лекции](https://gunenkov.notion.site/Transformer-Attention-is-all-you-need-cf0d5f5ecbae4203abc3a80e2f8e339a?pvs=4).

Генерировать спектрограммы можно с помощью **комбинации механизма внимания и сверточных нейронных сетей**, но здесь применяются более сложные модели, такие как Tacotron2 от NVIDIA.

![](https://i.imgur.com/HGlWMbp.png)

Воспользоваться предварительно обученным Tacotron2 (для английского языка) можно с помощью библиотеки torchtext.

## Практика

- [распознавание речи с использованием мелкепстральных коэффициентов](https://drive.google.com/file/d/1h_3aahliC6XA6QNLO79m9t1ol7r6Btma/view?usp=sharing); на всякий случай [ссылка на pdf](https://drive.google.com/file/d/1TzI3EBK52C1BpAdcbU7Zhlm_R6VRuo77/view?usp=sharing);
- [распознавание речи с использованием MEL-спектрограмм](https://drive.google.com/file/d/1dsX3a_Ccl2OqUOOf0g6LoOYa6pHF8LCX/view?usp=sharing); на всякий случай [ссылка на pdf](https://drive.google.com/file/d/1DobIlusRBrIIsdQm-ZJCcRQdqgygEThZ/view?usp=sharing);
- [коннекционистская временная классификация](https://sid2697.github.io/Blog_Sid/algorithm/2019/10/19/CTC-Loss.html); [пример распознавания речи на TensorFlow](https://keras.io/examples/audio/ctc_asr/).

А прямо здесь мы сейчас сделаем синтез речи с использованием модели из Hugging Face! Будем использовать пайплайны, чтобы было проще. В качестве модели возьмем [эту](https://huggingface.co/facebook/mms-tts-rus).

**Замечание:** я демонстрирую без GPU, но прямо настоятельно рекомендуется использовать видеокарту! Обращайте внимание, что результатом будет тензор, который для записи в файл или отображения в ноутбуке надо будет конвертировать в numpy-массив, предварительно перенеся на CPU.

In [ ]:
from transformers import pipeline
import scipy

In [ ]:
synthesiser = pipeline('text-to-speech', 'facebook/mms-tts-rus')
speech = synthesiser('Приятного аппетита тем, кто кушает! А тем, кто еще и пьет - РЕСПЕКТ')

Some weights of the model checkpoint at facebook/mms-tts-rus were not used when initializing VitsModel: ['posterior_encoder.wavenet.res_skip_layers.2.weight_g', 'posterior_encoder.wavenet.res_skip_layers.1.weight_g', 'flow.flows.3.wavenet.res_skip_layers.0.weight_g', 'flow.flows.0.wavenet.res_skip_layers.0.weight_v', 'posterior_encoder.wavenet.res_skip_layers.7.weight_g', 'flow.flows.0.wavenet.in_layers.2.weight_g', 'posterior_encoder.wavenet.in_layers.6.weight_v', 'flow.flows.1.wavenet.in_layers.1.weight_g', 'posterior_encoder.wavenet.res_skip_layers.13.weight_v', 'flow.flows.0.wavenet.res_skip_layers.2.weight_v', 'flow.flows.0.wavenet.res_skip_layers.1.weight_v', 'posterior_encoder.wavenet.in_layers.15.weight_v', 'flow.flows.0.wavenet.in_layers.3.weight_g', 'flow.flows.1.wavenet.in_layers.3.weight_g', 'posterior_encoder.wavenet.in_layers.14.weight_g', 'flow.flows.3.wavenet.res_skip_layers.3.weight_g', 'flow.flows.1.wavenet.res_skip_layers.2.weight_v', 'posterior_encoder.wavenet.res_s

In [ ]:
speech

{'audio': array([[-5.9142229e-05, -2.4576726e-05,  4.6246230e-05, ...,
         -7.1128321e-05, -8.6793079e-05, -3.2693300e-05]], dtype=float32),
 'sampling_rate': 16000}

In [ ]:
speech['audio']

array([[-5.9142229e-05, -2.4576726e-05,  4.6246230e-05, ...,
        -7.1128321e-05, -8.6793079e-05, -3.2693300e-05]], dtype=float32)

In [ ]:
from IPython.display import Audio

In [ ]:
Audio(speech['audio'], rate=speech['sampling_rate'])

In [ ]:
scipy.io.wavfile.write('respect.wav', speech['sampling_rate'], data=speech['audio'][0])

[Файл с результатом](https://drive.google.com/file/d/1bBS04WjaUn6O07HRyFUOpFqM7y8hODGo/view?usp=sharing).